# 3.2 — Correlation of water-balance component-wise PBIAS errors

The analysis quantifies whether annual precipitation and evapotranspiration bias errors vary together through time. The precipitation series is the annual pixelwise PBIAS of CONUS404 relative to PRISM. The evapotranspiration series is the annual pixelwise PBIAS of CONUS404 relative to Sanford WBET.

For each raster cell with at least the configured number of paired annual values, the analysis calculates Pearson's correlation coefficient:

$$
r=\frac{\sum_t(PB_{P,t}-\overline{PB}_P)(PB_{ET,t}-\overline{PB}_{ET})}{\sqrt{\sum_t(PB_{P,t}-\overline{PB}_P)^2}\sqrt{\sum_t(PB_{ET,t}-\overline{PB}_{ET})^2}}.
$$

Positive $r$ indicates that precipitation and ET PBIAS move in the same direction across years; negative $r$ indicates opposing temporal behavior. This calculation describes association between errors and does not imply causation.


## Annual PBIAS stacks and spatial alignment

The workflow uses the inclusive correlation period in `config.yml`. The workflow treats the ET PBIAS grid as the target grid and reprojected each precipitation PBIAS raster to the matching ET raster before concatenating the annual stacks.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import rioxarray
import xarray as xr
from rasterio.enums import Resampling

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from correlation_analysis import pearson_correlation
from accuracy_assessment import annual_pbias, load_comparison, load_config, paired_valid, write_raster
from regional_plots import plot_grouped_violins

config = load_config(REPOSITORY_ROOT / "config.yml")
correlation_config = config["correlation"]
period = config["study"]["correlation"]["annual"]
years = list(range(int(period["start_year"]), int(period["end_year"]) + 1))
precipitation_model = {
    "annual_dir": correlation_config["precipitation_model_dir"],
    "annual_pattern": correlation_config["precipitation_model_pattern"],
}
precipitation_reference = {
    "annual_dir": correlation_config["precipitation_reference_dir"],
    "annual_pattern": correlation_config["precipitation_reference_pattern"],
}
evapotranspiration_model = {
    "annual_dir": correlation_config["evapotranspiration_model_dir"],
    "annual_pattern": correlation_config["evapotranspiration_model_pattern"],
}
evapotranspiration_reference = {
    "annual_dir": correlation_config["evapotranspiration_reference_dir"],
    "annual_pattern": correlation_config["evapotranspiration_reference_pattern"],
}
precipitation_model_stack, precipitation_reference_stack = load_comparison(
    precipitation_model,
    precipitation_reference,
    "annual",
    years,
    resampling=config["processing"].get("resampling", "bilinear"),
)
et_model_stack, et_reference_stack = load_comparison(
    evapotranspiration_model,
    evapotranspiration_reference,
    "annual",
    years,
    resampling=config["processing"].get("resampling", "bilinear"),
)
positive_only = config["processing"].get("require_positive_values", True)
precipitation_model_stack, precipitation_reference_stack = paired_valid(
    precipitation_model_stack,
    precipitation_reference_stack,
    positive_only=positive_only,
)
et_model_stack, et_reference_stack = paired_valid(
    et_model_stack,
    et_reference_stack,
    positive_only=positive_only,
)
precipitation_pbias = annual_pbias(
    precipitation_model_stack,
    precipitation_reference_stack,
)
et_pbias = annual_pbias(et_model_stack, et_reference_stack)
resampling_method = getattr(
    Resampling,
    config["processing"].get("resampling", "bilinear"),
)
precipitation_stack = xr.concat(
    [
        precipitation_pbias.sel(year=year)
        .rio.reproject_match(et_pbias.sel(year=year), resampling=resampling_method)
        .expand_dims(year=[year])
        for year in years
    ],
    dim="year",
)
et_stack = et_pbias
print(f"Correlation period: {years[0]}–{years[-1]}")


## Pixelwise Pearson correlation

The workflow forms a paired finite-value mask independently at each cell and retains the actual number of valid annual pairs. It masks correlation where the valid count is below the configured minimum. It then applies the CONUS404-versus-Sanford ET nMAE raster as the spatial-domain mask and retains cells with 0–300% nMAE under the default configuration, reproducing the mask in the source analysis. The domain mask determines where the statistic is reported but does not enter the Pearson equation. The workflow retains exact zero correlation because it represents a valid absence of linear association.


In [ ]:
output_dir = Path(config["output_dir"]) / "correlation"
output_dir.mkdir(parents=True, exist_ok=True)
for year in years:
    write_raster(
        precipitation_stack.sel(year=year),
        output_dir / "component_pbias" / "precipitation_conus404_vs_prism" / f"wy{year}.tif",
    )
    write_raster(
        et_stack.sel(year=year),
        output_dir / "component_pbias" / "evapotranspiration_conus404_vs_sanford" / f"wy{year}.tif",
    )

correlation_metrics = pearson_correlation(
    precipitation_stack,
    et_stack,
    dim="year",
    min_valid=int(correlation_config["minimum_valid_years"]),
)
if correlation_config.get("domain_mask") is not None:
    domain = rioxarray.open_rasterio(
        correlation_config["domain_mask"], masked=True
    ).squeeze(drop=True)
    domain = domain.rio.reproject_match(correlation_metrics["pearson_r"])
    valid_domain = np.isfinite(domain)
    if correlation_config.get("domain_mask_minimum") is not None:
        valid_domain &= domain >= float(correlation_config["domain_mask_minimum"])
    if correlation_config.get("domain_mask_maximum") is not None:
        valid_domain &= domain <= float(correlation_config["domain_mask_maximum"])
    correlation_metrics = {
        name: data.where(valid_domain) for name, data in correlation_metrics.items()
    }

correlation_paths = {}
for name, data in correlation_metrics.items():
    path = output_dir / f"{name}.tif"
    write_raster(data, path)
    correlation_paths[name] = path
correlation_paths


## Climate-region correlation distributions

The workflow extracts the cellwise Pearson coefficients by climate region and summarized each distribution with its mean, median, sample standard deviation, and valid pixel count. The displayed range remained bounded by the mathematical interval −1 to 1.


In [ ]:
regions = gpd.read_file(config["regions"]["file"])
region_config = config["regions"]
figure_config = config["figures"]
correlation_stats = plot_grouped_violins(
    [("P–ET PBIAS Pearson r", correlation_paths["pearson_r"])],
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "pbias_pearson_correlation_violin.png",
    output_dir / "tables" / "pbias_pearson_correlation_regional_statistics.csv",
    "Pearson correlation (r)",
    tuple(figure_config["correlation"]["range"]),
    [figure_config["colors"][3]],
    dpi=figure_config["dpi"],
    exclude_zero=False,
)
correlation_stats
